In [16]:
%pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
%pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
%pip install --no-deps unsloth

In [17]:
from unsloth import FastVisionModel
import torch

In [21]:
from pathlib import Path
import sys
import textwrap

# Use current runtime workspace as project root (works in /content, local Jupyter, Kaggle).
project_root = Path.cwd().resolve()
(project_root / "config_loader").mkdir(parents=True, exist_ok=True)
(project_root / "configs").mkdir(parents=True, exist_ok=True)

# Ensure importable package: config_loader.load_config
init_file = project_root / "config_loader" / "__init__.py"
loader_file = project_root / "config_loader" / "config_loader.py"
if not init_file.exists():
    init_file.write_text("from .config_loader import load_config\n", encoding="utf-8")
if not loader_file.exists():
    loader_file.write_text(
        textwrap.dedent(
            '''
            from pathlib import Path
            import yaml

            def load_config(config_path: str = "configs/config.yaml"):
                path = Path(config_path)
                if not path.is_absolute():
                    path = Path.cwd() / path
                with path.open("r", encoding="utf-8") as f:
                    return yaml.safe_load(f)
            '''
        ).strip() + "\n",
        encoding="utf-8",
    )

# Ensure config file exists in this runtime.
config_file = project_root / "configs" / "config.yaml"
if not config_file.exists():
    config_file.write_text(
        textwrap.dedent(
            '''
            # Training configuration file
            
            modaname:
              model: "unsloth/Qwen2-VL-7B-Instruct"
              dataset: "unsloth/Latex_OCR"
            
            quantization:
              load_in_4bit: true
            
            LoRA:
              r: 8
              lora_alpha: 8
              lora_dropout: 0
              random_state: 3407
            
            training:
              batch_size: 2
              gradient_accumulation_steps: 4
              warmup_steps: 5
              max_steps: 30
              learning_rate: 2e-4
              weight_decay: 0.01
              optimizer: "adamw_8bit"
              logging_steps: 1
              dataset_num_proc: 2
              max_seq_length: 1024
            
            inference:
              max_new_tokens: 128
              temperature: 1.5
              min_p: 0.1
            '''
        ).lstrip(),
        encoding="utf-8",
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config_loader import load_config
config = load_config(config_path="configs/config.yaml")
print(f"Loaded config from: {config_file}")

Loaded config from: /content/configs/config.yaml


In [23]:
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"
]

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    load_in_4bit=config["quantization"]["load_in_4bit"],
    use_gradient_checkpointing="unsloth",
)

==((====))==  Unsloth 2026.4.6: Fast Qwen2_Vl patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

In [ ]:
quantization_config = config["quantization"]
modaname_config = config["modaname"]
LoRA_config = config["LoRA"]
training_config = config["training"]
inference_config = config["inference"]

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=False,

    r=LoRA_config["r"],
    lora_alpha=LoRA_config["lora_alpha"],
    lora_dropout=LoRA_config["lora_dropout"],
    bias=LoRA_config["bias"],
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset(modaname_config["dataset"], split="train")

In [ ]:
instruction = "Your duty is to provide the LaTex representation for this image."

In [ ]:
def convert_to_conversation(sample):
  conversation = [
      {"role": "user",
       "content": [
           {"type": "text", "text": instruction},
           {"type": "image", "image": sample["image"]}
       ]
       },
      {"role": "assistant",
       "content": [
           {"type": "text", "text": sample["text"]}
       ]
       }
  ]
  return {"messages": conversation}

In [ ]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

In [ ]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

In [ ]:
FastVisionModel.for_training(model) 

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2VLForConditionalGeneration(
      (model): Qwen2VLModel(
        (visual): Qwen2VisionTransformerPretrainedModel(
          (patch_embed): PatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2VLVisionBlock(
              (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
              (attn): VisionAttention(
                (qkv): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1280, out_features=3840, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1280, out_features=8, bi

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=converted_dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=training_config["warmup_steps"],
        max_steps=training_config["max_steps"],
        learning_rate=training_config["learning_rate"],
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=training_config["logging_steps"],
        optim=training_config["optimizer"],
        weight_decay=training_config["weight_decay"],
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=training_config["dataset_num_proc"],
        max_seq_length=training_config["max_seq_length"],
    ),
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3018055686.py, line 9)

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 68,686 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 7,012,352 of 8,298,387,968 (0.08% trained)


Step,Training Loss
1,1.296083
2,1.440867
3,1.563911
4,1.218124
5,1.266203
6,1.444150
7,1.387058
8,1.083676
9,0.916504
10,1.074443


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=30, training_loss=0.7973099519809087, metrics={'train_runtime': 291.5252, 'train_samples_per_second': 0.823, 'train_steps_per_second': 0.103, 'total_flos': 1398680649234432.0, 'train_loss': 0.7973099519809087})

/content/config_loader/config_loader.py: False
/content/config_loader.py: False
/content/configs/config.yaml: False
configs/config.yaml: False
